In [1]:
import os

import numpy as np
import pandas as pd

from sklearn import metrics

import pickle

In [2]:
dataset_dir = "dataset"
models_dir = "models"

In [3]:
data_df = pd.read_csv(os.path.join(dataset_dir, "test_set.csv"))
X_test = data_df.drop(columns=["TARGET"])
y_test = data_df["TARGET"]

## Logistic regression

Load the model.

In [4]:
model_path = os.path.join(models_dir, "lr_pipe.pkl")
with open(model_path, "rb") as file:
    lr_pipe = pickle.load(file)

In [5]:
y_pred = lr_pipe.predict(X_test)

In [6]:
print(metrics.classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.96      0.69      0.80     55129
         1.0       0.16      0.67      0.26      4871

    accuracy                           0.69     60000
   macro avg       0.56      0.68      0.53     60000
weighted avg       0.89      0.69      0.76     60000



In [8]:
print(f"ROC AUC Score: {metrics.roc_auc_score(y_test, y_pred)}")

ROC AUC Score: 0.6793098301801677


## XGBoost

In [9]:
model_path = os.path.join(models_dir, "xgb_clf.pkl")
with open(model_path, "rb") as file:
    xgb_clf = pickle.load(file)

In [10]:
y_pred = xgb_clf.predict(X_test)

In [11]:
print(metrics.classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.94      0.91      0.92     55129
         1.0       0.22      0.29      0.25      4871

    accuracy                           0.86     60000
   macro avg       0.58      0.60      0.58     60000
weighted avg       0.88      0.86      0.87     60000



In [12]:
print(f"ROC AUC Score: {metrics.roc_auc_score(y_test, y_pred)}")

ROC AUC Score: 0.5981537548934468


## Fully-connected neural network

In [13]:
import torch
import torch.nn as nn
from torch.nn import functional as F

Define the model class.

In [14]:
BATCH_SIZE = 1024
DROPOUT = 0.3

In [15]:
class NNBinaryClassifier(nn.Module):
    def __init__(self, in_features, dropout=0.0):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Linear(in_features=in_features, out_features=128),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(in_features=128, out_features=64),
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(in_features=64, out_features=1)
        )

    def forward(self, x):
        return self.seq(x)

Load the scaler

In [16]:
scaler_path = os.path.join(models_dir, "scaler.pkl")
with open(scaler_path, "rb") as f:
    scaler = pickle.load(f)

X_test_t = scaler.transform(X_test)

In [17]:
X_test_t = torch.tensor(X_test_t, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

test_dataset = torch.utils.data.TensorDataset(X_test_t, y_test_t)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)

Loading the model.

In [18]:
num_features = X_test_t.shape[1]

In [19]:
model_path = os.path.join(models_dir, "fcnn.pt")
fcnn = NNBinaryClassifier(in_features=num_features, dropout=DROPOUT)
fcnn.load_state_dict(torch.load(model_path, weights_only=True))

<All keys matched successfully>

In [22]:
fcnn.eval()
with torch.no_grad():
    preds = []
    targets = []
    for X_batch, y_batch in test_dataloader:
        y_logits = fcnn(X_batch)
        y_pred = (torch.sigmoid(y_logits) > 0.5).to(dtype=torch.float32)

        preds.extend(y_pred)
        targets.extend(y_batch.tolist())

    preds = np.array([s.item() for s in preds])
    targets = np.array(targets).flatten()

    print(metrics.classification_report(targets, preds))
    print(f"ROC AUC Score: {metrics.roc_auc_score(targets, preds)}")

              precision    recall  f1-score   support

         0.0       0.96      0.68      0.80     55129
         1.0       0.16      0.67      0.26      4871

    accuracy                           0.68     60000
   macro avg       0.56      0.68      0.53     60000
weighted avg       0.89      0.68      0.76     60000

ROC AUC Score: 0.6768041508019865
